# Assessment 2 - Financial Accounting & GL Reconciliation

See `docs/milestones.md` and `docs/design/assignment.md` for task scope.
Connectivity conventions: see `00_template_connectivity_check.ipynb`.


In [1]:
import os
from pyspark.sql import SparkSession
import psycopg2

POSTGRES_DB = os.environ["POSTGRES_DB"]
POSTGRES_USER = os.environ["POSTGRES_USER"]
POSTGRES_PASSWORD = os.environ["POSTGRES_PASSWORD"]


## Task 1 - GL Integrity and Reconciliation

See `results/assessment-2/assessment-2-overview.md` for scenario, table shapes, and scale.

**10.CK.01**-**10.CK.08** (task refs `01.01`-`01.08`) are implemented below: the GL's own arithmetic
integrity, an independent recomputation of debit/credit movements from `bronze.finance_transactions`,
and a per-dimension reconciliation across legal entity, GL account, cost center, currency, and
accounting date.


In [2]:
spark = (
    SparkSession.builder.master("spark://spark-master:7077")
    .appName("assessment2-task1-gl-integrity")
    .getOrCreate()
)


def jdbc_table(table_name):
    return spark.read.jdbc(
        url=f"jdbc:postgresql://postgres:5432/{POSTGRES_DB}",
        table=table_name,
        properties={
            "user": POSTGRES_USER,
            "password": POSTGRES_PASSWORD,
            "driver": "org.postgresql.Driver",
        },
    )


gl_df = jdbc_table("finance.gl_balance")
txn_df = jdbc_table("bronze.finance_transactions")

print(f"[INFO] finance.gl_balance row_count={gl_df.count()}")
print(f"[INFO] bronze.finance_transactions row_count={txn_df.count()}")


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/09/08 08:20:52 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


[INFO] finance.gl_balance row_count=589


[INFO] bronze.finance_transactions row_count=1523


### 10.CK.01 - arithmetic integrity

`opening_balance + debit_movement - credit_movement = closing_balance` on every `finance.gl_balance`
row. Tolerance: exact equality - all four columns are `decimal(20,2)` and the expression is pure
addition/subtraction, so a nonzero result is a genuine arithmetic break, not a rounding artifact.


In [3]:
from pyspark.sql.functions import col, sum as spark_sum, abs as spark_abs

# kept as native decimal(20,2) here (no cast to double) - the arithmetic check below relies on
# exact equality, and a double cast introduces float rounding noise (observed ~1e-12 on a first
# pass, inflating 5 true violations to 203 false ones); double is safe further down for the
# tolerance-based (0.01) recomputation checks, so the cast happens there instead.
gl_exact = gl_df.select(
    col("accounting_date"), col("legal_entity"), col("gl_account"),
    col("cost_center"), col("currency"),
    col("opening_balance"), col("debit_movement"), col("credit_movement"), col("closing_balance"),
)

arithmetic_check = gl_exact.withColumn(
    "computed_closing", col("opening_balance") + col("debit_movement") - col("credit_movement")
).withColumn(
    "variance", col("closing_balance") - col("computed_closing")
)

arithmetic_violations = arithmetic_check.filter(col("variance") != 0)
violation_count = arithmetic_violations.count()

print(f"[INFO] 10.CK.01 arithmetic_integrity_violations={violation_count}")
arithmetic_violations.select(
    "accounting_date", "legal_entity", "gl_account", "cost_center", "currency",
    "closing_balance", "computed_closing", "variance",
).show(20, truncate=False)


[INFO] 10.CK.01 arithmetic_integrity_violations=5


+---------------+------------+----------+-----------+--------+---------------+----------------+--------+
|accounting_date|legal_entity|gl_account|cost_center|currency|closing_balance|computed_closing|variance|
+---------------+------------+----------+-----------+--------+---------------+----------------+--------+
|2026-08-17     |LE3         |GL1014    |CC05       |SGD     |37363.87       |39698.69        |-2334.82|
|2026-08-17     |LE4         |GL1007    |CC08       |USD     |-11176.81      |-9612.03        |-1564.78|
|2026-08-20     |LE1         |GL1003    |CC04       |SGD     |82746.17       |79671.14        |3075.03 |
|2026-08-21     |LE4         |GL1009    |CC10       |USD     |85752.25       |83837.66        |1914.59 |
|2026-08-21     |LE4         |GL1013    |CC04       |SGD     |169471.57      |172377.24       |-2905.67|
+---------------+------------+----------+-----------+--------+---------------+----------------+--------+



### 10.CK.02 / 10.CK.03 - independent movement recomputation

**business key & date convention** - `finance.gl_balance`'s five-dimension grouping key
(`accounting_date, legal_entity, gl_account, cost_center, currency`) is also the grouping key this
recomputation aggregates `bronze.finance_transactions` onto, joining `bronze.finance_transactions.posting_date`
to `gl_balance.accounting_date` - the GL is dated by *posting*, not by `transaction_date`.

Tolerance: `MOVEMENT_TOLERANCE_ABS = 0.01` (one minor-currency-unit) applied independently to each side.
A full outer join (not left/inner) so a GL key with no matching transactions, or a transaction key with
no matching GL row, both surface as a variance instead of silently dropping out of the comparison.


In [4]:
from pyspark.sql.functions import when

MOVEMENT_TOLERANCE_ABS = 0.01

# double is safe here - comparisons below use a 0.01 tolerance, well above any float rounding noise.
gl = gl_df.select(
    col("accounting_date"), col("legal_entity"), col("gl_account"),
    col("cost_center"), col("currency"),
    col("debit_movement").cast("double").alias("debit_movement"),
    col("credit_movement").cast("double").alias("credit_movement"),
)

txn = txn_df.select(
    col("posting_date").alias("accounting_date"),
    col("legal_entity"), col("gl_account"), col("cost_center"), col("currency"),
    col("debit_credit_indicator"),
    col("local_amount").cast("double").alias("local_amount"),
)

recomputed = txn.groupBy("accounting_date", "legal_entity", "gl_account", "cost_center", "currency").agg(
    spark_sum(when(col("debit_credit_indicator") == "DEBIT", col("local_amount")).otherwise(0.0)).alias("recomputed_debit"),
    spark_sum(when(col("debit_credit_indicator") == "CREDIT", col("local_amount")).otherwise(0.0)).alias("recomputed_credit"),
)

JOIN_KEYS = ["accounting_date", "legal_entity", "gl_account", "cost_center", "currency"]

movement_compare = gl.join(recomputed, JOIN_KEYS, "full_outer").select(
    *[col(k) for k in JOIN_KEYS],
    col("debit_movement"), col("recomputed_debit"),
    col("credit_movement"), col("recomputed_credit"),
).fillna(0.0, subset=["debit_movement", "recomputed_debit", "credit_movement", "recomputed_credit"]).withColumn(
    "debit_variance", col("debit_movement") - col("recomputed_debit")
).withColumn(
    "credit_variance", col("credit_movement") - col("recomputed_credit")
)

movement_variances = movement_compare.filter(
    (spark_abs(col("debit_variance")) > MOVEMENT_TOLERANCE_ABS)
    | (spark_abs(col("credit_variance")) > MOVEMENT_TOLERANCE_ABS)
)
movement_compare.cache()
movement_variance_count = movement_variances.count()

print(f"[INFO] 10.CK.02/10.CK.03 movement_recomputation_variances={movement_variance_count}")
movement_variances.orderBy(spark_abs(col("debit_variance") + col("credit_variance")).desc()).show(20, truncate=False)


[INFO] 10.CK.02/10.CK.03 movement_recomputation_variances=284


+---------------+------------+----------+-----------+--------+--------------+------------------+---------------+------------------+-------------------+-------------------+
|accounting_date|legal_entity|gl_account|cost_center|currency|debit_movement|recomputed_debit  |credit_movement|recomputed_credit |debit_variance     |credit_variance    |
+---------------+------------+----------+-----------+--------+--------------+------------------+---------------+------------------+-------------------+-------------------+
|2026-08-19     |LE2         |GL9999    |CC99       |EUR     |39440.74      |57310.36          |22662.42       |32930.85          |-17869.620000000003|-10268.43          |
|2026-08-20     |LE3         |GL9999    |CC99       |USD     |26677.86      |35880.07          |50026.3        |66845.22          |-9202.21           |-16818.92          |
|2026-08-19     |LE4         |GL9999    |CC99       |USD     |868.14        |1156.61           |72566.0        |96737.51          |-288.4699

### 10.CK.04 - 10.CK.08 - dimensional reconciliation

The same recomputation, rolled up to one dimension at a time instead of the full five-key grain.
`reconciliation_status` per row: `PASS` if `variance_pct < 0.1%`, `WARNING` if `< 1%`, `FAIL` otherwise -
the same thresholds `reconciliation.rc_batch_control.status` already fixes.


In [5]:
def status_for(variance_pct):
    pct = abs(variance_pct)
    if pct < 0.1:
        return "PASS"
    if pct < 1.0:
        return "WARNING"
    return "FAIL"


LEVEL_DIMENSIONS = [
    ("10.CK.04", "legal_entity"),
    ("10.CK.05", "gl_account"),
    ("10.CK.06", "cost_center"),
    ("10.CK.07", "currency"),
    ("10.CK.08", "accounting_date"),
]

dimension_summaries = {}

for check_id, dim in LEVEL_DIMENSIONS:
    rolled = movement_compare.groupBy(dim).agg(
        spark_sum("debit_movement").alias("gl_debit"),
        spark_sum("recomputed_debit").alias("recomputed_debit"),
        spark_sum("credit_movement").alias("gl_credit"),
        spark_sum("recomputed_credit").alias("recomputed_credit"),
    ).withColumn(
        "debit_variance", col("gl_debit") - col("recomputed_debit")
    ).withColumn(
        "credit_variance", col("gl_credit") - col("recomputed_credit")
    ).collect()

    rows = []
    for r in rolled:
        gl_total = (r["gl_debit"] or 0.0) + (r["gl_credit"] or 0.0)
        var_total = (r["debit_variance"] or 0.0) + (r["credit_variance"] or 0.0)
        variance_pct = round((abs(var_total) / gl_total * 100) if gl_total else 0.0, 4)
        rows.append({
            "group_value": r[dim], "gl_debit": r["gl_debit"], "gl_credit": r["gl_credit"],
            "debit_variance": r["debit_variance"], "credit_variance": r["credit_variance"],
            "variance_pct": variance_pct, "status": status_for(variance_pct),
        })
    dimension_summaries[dim] = rows
    worst = sorted(rows, key=lambda r: abs(r["variance_pct"]), reverse=True)[:5]
    print(f"[INFO] {check_id} dimensional reconciliation by {dim} - top variance groups:")
    for r in worst:
        print(f"  [{r['status']}] {dim}={r['group_value']} debit_variance={r['debit_variance']:.2f} "
              f"credit_variance={r['credit_variance']:.2f} variance_pct={r['variance_pct']}%")


[INFO] 10.CK.04 dimensional reconciliation by legal_entity - top variance groups:
  [FAIL] legal_entity=LE4 debit_variance=-230710.45 credit_variance=-214277.49 variance_pct=12.2325%
  [FAIL] legal_entity=LE3 debit_variance=-202704.38 credit_variance=-255233.01 variance_pct=11.4772%
  [FAIL] legal_entity=LE2 debit_variance=-209941.98 credit_variance=-173487.86 variance_pct=11.3555%
  [FAIL] legal_entity=LE1 debit_variance=-205855.71 credit_variance=-199604.04 variance_pct=9.4227%


[INFO] 10.CK.05 dimensional reconciliation by gl_account - top variance groups:
  [FAIL] gl_account=GL1009 debit_variance=-116933.09 credit_variance=0.00 variance_pct=14.3507%
  [FAIL] gl_account=GL1004 debit_variance=-113084.97 credit_variance=0.00 variance_pct=13.2607%
  [FAIL] gl_account=GL1007 debit_variance=0.00 credit_variance=-113767.37 variance_pct=12.5202%
  [FAIL] gl_account=GL1013 debit_variance=-92280.61 credit_variance=0.00 variance_pct=11.8572%
  [FAIL] gl_account=GL1011 debit_variance=-98341.46 credit_variance=0.00 variance_pct=11.7118%


[INFO] 10.CK.06 dimensional reconciliation by cost_center - top variance groups:
  [FAIL] cost_center=CC10 debit_variance=-116933.09 credit_variance=0.00 variance_pct=14.3507%
  [FAIL] cost_center=CC08 debit_variance=0.00 credit_variance=-111961.34 variance_pct=12.6558%
  [FAIL] cost_center=CC05 debit_variance=-182485.01 credit_variance=-2712.39 variance_pct=11.4337%
  [FAIL] cost_center=CC99 debit_variance=-90289.96 credit_variance=-340405.49 variance_pct=11.0748%
  [FAIL] cost_center=CC01 debit_variance=-75094.56 credit_variance=-96062.97 variance_pct=10.8393%


[INFO] 10.CK.07 dimensional reconciliation by currency - top variance groups:
  [FAIL] currency=EUR debit_variance=-325397.67 credit_variance=-331201.88 variance_pct=45.7804%
  [FAIL] currency=USD debit_variance=-523814.85 credit_variance=-511400.52 variance_pct=34.1154%
  [PASS] currency=SGD debit_variance=0.00 credit_variance=0.00 variance_pct=0.0%


[INFO] 10.CK.08 dimensional reconciliation by accounting_date - top variance groups:
  [FAIL] accounting_date=2026-08-19 debit_variance=-213002.57 credit_variance=-151971.43 variance_pct=11.8745%
  [FAIL] accounting_date=2026-08-21 debit_variance=-148435.87 credit_variance=-213536.53 variance_pct=11.6854%
  [FAIL] accounting_date=2026-08-18 debit_variance=-169638.78 credit_variance=-144322.85 variance_pct=10.7357%
  [FAIL] accounting_date=2026-08-20 debit_variance=-173426.88 credit_variance=-154774.23 variance_pct=10.4985%
  [FAIL] accounting_date=2026-08-17 debit_variance=-144708.42 credit_variance=-177997.36 variance_pct=10.4588%


### write results to `reconciliation.rc_*`

`reconciliation.rc_reconciliation_results.dimension` is a closed set (`row_count`, `amount`) fixed by
feature 05; widening it is a schema change out of this tracker's scope - the same constraint
[09](../features/../assessments/09-as01-data-profiling-reconciliation.md) hit for its own level 1 batch
totals. `row_count` compares `bronze.finance_transactions` count against `finance.gl_balance` count;
`amount` compares `SUM(local_amount)` against `SUM(closing_balance)` - the two GL-integrity totals this
task can express against the existing schema. The full per-dimension detail above is reported in the
notebook and the reconciliation-results deliverable only.


In [6]:
def reserve_batch_id(conn):
    with conn.cursor() as cur:
        cur.execute("SELECT nextval('reconciliation.rc_batch_control_batch_id_seq');")
        return cur.fetchone()[0]


def insert_batch_control(conn, batch_id, status):
    with conn.cursor() as cur:
        cur.execute(
            "INSERT INTO reconciliation.rc_batch_control (batch_id, batch_date, assessment_id, status) "
            "VALUES (%s, CURRENT_DATE, %s, %s)",
            (batch_id, "assessment-2", status),
        )


def update_batch_status(conn, batch_id, status):
    with conn.cursor() as cur:
        cur.execute(
            "UPDATE reconciliation.rc_batch_control SET status = %s WHERE batch_id = %s",
            (status, batch_id),
        )


def insert_result_row(conn, batch_id, dimension, source_value, target_value):
    variance = target_value - source_value
    variance_pct = round((variance / source_value * 100) if source_value else 0.0, 4)
    status = status_for(variance_pct)
    with conn.cursor() as cur:
        cur.execute(
            "INSERT INTO reconciliation.rc_reconciliation_results "
            "(batch_id, dimension, source_value, target_value, variance, variance_pct, reconciliation_status) "
            "VALUES (%s, %s, %s, %s, %s, %s, %s)",
            (batch_id, dimension, source_value, target_value, variance, variance_pct, status),
        )
    return status


txn_count = txn_df.count()
gl_count = gl_df.count()
txn_amount = txn_df.agg(spark_sum(col("local_amount").cast("double"))).collect()[0][0] or 0.0
gl_amount = gl_df.agg(spark_sum(col("closing_balance").cast("double"))).collect()[0][0] or 0.0

conn = psycopg2.connect(host="postgres", port=5432, dbname=POSTGRES_DB, user=POSTGRES_USER, password=POSTGRES_PASSWORD)
batch_id = reserve_batch_id(conn)
insert_batch_control(conn, batch_id, "RUNNING")

statuses = [
    insert_result_row(conn, batch_id, "row_count", txn_count, gl_count),
    insert_result_row(conn, batch_id, "amount", txn_amount, gl_amount),
]
overall_status = max(statuses, key=lambda s: {"PASS": 0, "WARNING": 1, "FAIL": 2}[s])
update_batch_status(conn, batch_id, overall_status)
conn.commit()
conn.close()

print(f"[INFO] source(bronze.finance_transactions): count={txn_count} amount={txn_amount:.2f}")
print(f"[INFO] target(finance.gl_balance): count={gl_count} amount={gl_amount:.2f}")
print(f"[{overall_status}] assessment2-task1-gl-integrity: batch_id={batch_id}")


[INFO] source(bronze.finance_transactions): count=1523 amount=16999151.01
[INFO] target(finance.gl_balance): count=589 amount=12996847.89
[FAIL] assessment2-task1-gl-integrity: batch_id=12


In [7]:
spark.stop()
